# JoeyLLM — Australia 5B Single-File Stratified Sample Notebook

**Notebook version.**

Goal: create **one huge Parquet file** containing about **5 billion tokens** from the Australian FineWeb/Common Crawl data.

Default input:

```text
/home/jovyan/data/Australia
```

Default output dataset file:

```text
/home/jovyan/data/australia_5B_single_file/australia_5B_sample_single_file.parquet
```

Default stats file:

```text
/home/jovyan/data/australia_5B_single_file/australia_5B_sampling_stats.json
```

⚠️ **Do not push the generated Parquet dataset to GitHub.** Only push this notebook, scripts, README/configs, and small stats/log files.

In [ ]:
# Optional, run only if packages are missing
# !pip install -U pyarrow tqdm numpy huggingface_hub datasets

In [ ]:
# ── Cell 1: Imports & configuration ──────────────────────────────────────────
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

DATA_ROOT = Path('/home/jovyan/data/Australia')
OUTPUT_DIR = Path('/home/jovyan/data/australia_5B_single_file')
OUTPUT_FILE = OUTPUT_DIR / 'australia_5B_sample_single_file.parquet'
STATS_FILE = OUTPUT_DIR / 'australia_5B_sampling_stats.json'

TARGET_TOKENS = 5_000_000_000
RANDOM_SEED = 42
COMPRESSION = 'snappy'  # use 'zstd' for smaller file, 'snappy' for speed
OVERWRITE = False

random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT     : {DATA_ROOT}')
print(f'OUTPUT_FILE   : {OUTPUT_FILE}')
print(f'STATS_FILE    : {STATS_FILE}')
print(f'TARGET_TOKENS : {TARGET_TOKENS:,}')
print(f'RANDOM_SEED   : {RANDOM_SEED}')
print(f'COMPRESSION   : {COMPRESSION}')

In [ ]:
# ── Cell 2: Helper functions ─────────────────────────────────────────────────

def read_progress_tokens(progress_path: Path) -> int:
    """Read total_tokens from one progress.json file."""
    try:
        with progress_path.open('r', encoding='utf-8') as f:
            info = json.load(f)
        return int(info.get('total_tokens', 0) or 0)
    except Exception:
        return 0


def scan_dataset(data_root: Path):
    """Scan CC-MAIN folders and collect parquet files plus per-folder token metadata."""
    if not data_root.exists():
        raise FileNotFoundError(f'DATA_ROOT does not exist: {data_root}')

    dir_files = {}
    dir_tokens = {}

    for batch_dir in sorted(data_root.iterdir()):
        if not batch_dir.is_dir() or not batch_dir.name.startswith('CC-MAIN'):
            continue

        files = sorted(batch_dir.glob('*.parquet'))
        if not files:
            continue

        dir_files[batch_dir.name] = files
        dir_tokens[batch_dir.name] = read_progress_tokens(batch_dir / 'progress.json')

    if not dir_files:
        raise RuntimeError(f'No CC-MAIN parquet files found under: {data_root}')

    missing = [d for d, t in dir_tokens.items() if t <= 0]
    if missing:
        known = [t for t in dir_tokens.values() if t > 0]
        median_known = int(np.median(known)) if known else 1
        for d in missing:
            dir_tokens[d] = max(1, median_known)
        print(
            f'[WARN] {len(missing)} folders do not have valid progress.json total_tokens; '
            'using median token estimate for quota allocation.'
        )

    return dir_files, dir_tokens


def allocate_token_quotas(dir_tokens: dict, target_tokens: int) -> dict:
    """
    Allocate target tokens to each CC-MAIN folder proportional to its available token count.
    This keeps the sample stratified across Common Crawl batches.
    """
    total_tokens = sum(dir_tokens.values())
    if total_tokens <= 0:
        raise RuntimeError('Could not determine total token count from progress.json files.')

    raw = {d: target_tokens * (t / total_tokens) for d, t in dir_tokens.items()}
    quotas = {d: int(math.floor(v)) for d, v in raw.items()}

    remainder = target_tokens - sum(quotas.values())
    if remainder > 0:
        order = sorted(raw.keys(), key=lambda d: raw[d] - quotas[d], reverse=True)
        for d in order[:remainder]:
            quotas[d] += 1

    for d in quotas:
        quotas[d] = max(1, quotas[d])

    return quotas


def get_token_counts(table: pa.Table) -> np.ndarray:
    """Return token_count column as numpy array."""
    if 'token_count' not in table.column_names:
        raise KeyError('Required column not found: token_count')
    return table.column('token_count').combine_chunks().to_numpy(zero_copy_only=False)

In [ ]:
# ── Cell 3: Scan dataset and calculate quotas ────────────────────────────────
dir_files, dir_tokens = scan_dataset(DATA_ROOT)
all_dirs = sorted(dir_files.keys())

total_files = sum(len(files) for files in dir_files.values())
total_tokens_available = int(sum(dir_tokens.values()))
sample_ratio = TARGET_TOKENS / total_tokens_available
quota_per_dir = allocate_token_quotas(dir_tokens, TARGET_TOKENS)

print(f'CC-MAIN folders       : {len(all_dirs):,}')
print(f'Parquet files         : {total_files:,}')
print(f'Available tokens      : {total_tokens_available:,}')
print(f'Target sample ratio   : {sample_ratio:.4%}')
print(f'Quota sum             : {sum(quota_per_dir.values()):,}')

print('\nTop 10 folders by token quota:')
for d in sorted(all_dirs, key=lambda x: quota_per_dir[x], reverse=True)[:10]:
    print(
        f'  {d:<20} files={len(dir_files[d]):>5} '
        f'available={dir_tokens[d]:>15,} quota={quota_per_dir[d]:>13,}'
    )

In [ ]:
# ── Cell 4: Main sampling loop — write ONE huge parquet file ─────────────────
if OUTPUT_FILE.exists() and not OVERWRITE:
    raise FileExistsError(
        f'Output file already exists: {OUTPUT_FILE}\n'
        'Set OVERWRITE = True in Cell 1 if you really want to replace it.'
    )

rng = random.Random(RANDOM_SEED)
compression = None if COMPRESSION == 'none' else COMPRESSION

writer = None
stats_per_dir = defaultdict(lambda: {'files': 0, 'row_groups': 0, 'tokens': 0, 'rows': 0})

total_tokens_written = 0
total_rows_written = 0
files_touched = 0
row_groups_touched = 0
t0 = time.time()

pbar = tqdm(total=TARGET_TOKENS, unit='tok', unit_scale=True, desc='Sampling progress', mininterval=1.0)

try:
    for dir_name in all_dirs:
        if total_tokens_written >= TARGET_TOKENS:
            break

        dir_quota = quota_per_dir[dir_name]
        dir_collected = 0

        files = list(dir_files[dir_name])
        rng.shuffle(files)

        for fp in files:
            if dir_collected >= dir_quota or total_tokens_written >= TARGET_TOKENS:
                break

            try:
                parquet_file = pq.ParquetFile(fp)
            except Exception as e:
                print(f'[WARN] Cannot open {fp}: {e}')
                continue

            file_used = False

            for rg_idx in range(parquet_file.num_row_groups):
                if dir_collected >= dir_quota or total_tokens_written >= TARGET_TOKENS:
                    break

                try:
                    table = parquet_file.read_row_group(rg_idx)
                except Exception as e:
                    print(f'[WARN] Cannot read {fp.name}, row_group={rg_idx}: {e}')
                    continue

                counts = get_token_counts(table)
                if len(counts) == 0:
                    continue

                cumsum = counts.cumsum()
                need_dir = dir_quota - dir_collected
                need_global = TARGET_TOKENS - total_tokens_written
                need = min(need_dir, need_global)

                if cumsum[-1] <= need:
                    take_n = len(counts)
                    take_tokens = int(cumsum[-1])
                else:
                    take_n = int(np.searchsorted(cumsum, need, side='left') + 1)
                    take_tokens = int(cumsum[take_n - 1])
                    table = table.slice(0, take_n)

                if writer is None:
                    writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression=compression)

                writer.write_table(table)

                dir_collected += take_tokens
                total_tokens_written += take_tokens
                total_rows_written += take_n
                row_groups_touched += 1
                file_used = True

                stats_per_dir[dir_name]['row_groups'] += 1
                stats_per_dir[dir_name]['tokens'] += take_tokens
                stats_per_dir[dir_name]['rows'] += take_n

                pbar.update(take_tokens)

                if row_groups_touched % 25 == 0:
                    elapsed = time.time() - t0
                    speed = total_tokens_written / max(elapsed, 1e-6)
                    print(
                        f'\n[{row_groups_touched} row groups] '
                        f'written={total_tokens_written/1e9:.3f}B tokens, '
                        f'dirs={len(stats_per_dir)}/{len(all_dirs)}, '
                        f'speed={speed/1e6:.1f}M tok/s'
                    )

                del table, counts, cumsum

            if file_used:
                files_touched += 1
                stats_per_dir[dir_name]['files'] += 1

        if dir_collected < dir_quota:
            print(
                f'[WARN] {dir_name} under-filled: '
                f'got {dir_collected:,} / quota {dir_quota:,} tokens'
            )

finally:
    if writer is not None:
        writer.close()
    pbar.close()

elapsed_total = time.time() - t0
output_size_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024 if OUTPUT_FILE.exists() else 0

print('\n✅ Australia 5B single-file sampling complete')
print(f'Output parquet file  : {OUTPUT_FILE}')
print(f'Total tokens written : {total_tokens_written:,}')
print(f'Total rows written   : {total_rows_written:,}')
print(f'Files touched        : {files_touched:,}')
print(f'Row groups touched   : {row_groups_touched:,}')
print(f'Directories covered  : {len(stats_per_dir):,} / {len(all_dirs):,}')
print(f'Output size          : {output_size_mb:.1f} MB')
print(f'Elapsed              : {elapsed_total/60:.1f} minutes')

In [ ]:
# ── Cell 5: Save stats report ────────────────────────────────────────────────
stats = {
    'dataset': 'Australia',
    'output_mode': 'single_parquet_file',
    'sampling_method': 'stratified_by_cc_main_batch_proportional_token_quota_single_file',
    'data_root': str(DATA_ROOT),
    'output_file': str(OUTPUT_FILE),
    'stats_file': str(STATS_FILE),
    'random_seed': RANDOM_SEED,
    'compression': COMPRESSION,
    'target_tokens': int(TARGET_TOKENS),
    'total_available_tokens_from_progress_json': int(total_tokens_available),
    'sample_ratio': sample_ratio,
    'total_tokens_written': int(total_tokens_written),
    'total_rows_written': int(total_rows_written),
    'files_touched': int(files_touched),
    'row_groups_touched': int(row_groups_touched),
    'dirs_covered': len(stats_per_dir),
    'total_dirs': len(all_dirs),
    'elapsed_minutes': round(elapsed_total / 60, 2),
    'average_speed_tokens_per_second': round(total_tokens_written / max(elapsed_total, 1e-6), 2),
    'output_size_mb': round(output_size_mb, 2),
    'quota_per_directory': {k: int(quota_per_dir[k]) for k in sorted(quota_per_dir)},
    'per_directory': {k: v for k, v in sorted(stats_per_dir.items())},
}

with STATS_FILE.open('w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f'Stats saved to: {STATS_FILE}')

In [ ]:
# ── Cell 6: Lightweight output verification ──────────────────────────────────
# Do NOT use pq.read_table(OUTPUT_FILE) here; the 5B file can be huge.
# This only reads Parquet metadata, not the full dataset.

pf = pq.ParquetFile(OUTPUT_FILE)
print(f'Output file exists     : {OUTPUT_FILE.exists()}')
print(f'Output file size MB    : {OUTPUT_FILE.stat().st_size / 1024 / 1024:.1f}')
print(f'Parquet row groups     : {pf.num_row_groups:,}')
print(f'Parquet schema columns : {pf.schema_arrow.names}')

print('\nStats token check:')
print(f'  target_tokens        : {stats["target_tokens"]:,}')
print(f'  total_tokens_written : {stats["total_tokens_written"]:,}')
print(f'  difference           : {stats["total_tokens_written"] - stats["target_tokens"]:,}')

In [ ]:
# ── Cell 7: Hugging Face upload template ─────────────────────────────────────
# Run only after the output file has been created and verified.
# Keep the dataset file outside GitHub; upload it to Hugging Face instead.

# !pip install -U huggingface_hub
# !huggingface-cli login

# from huggingface_hub import HfApi
# api = HfApi()
# repo_id = 'YOUR_HF_USERNAME_OR_ORG/joeyllm-australia-5b'
# api.create_repo(repo_id=repo_id, repo_type='dataset', exist_ok=True)
# api.upload_file(
#     path_or_fileobj=str(OUTPUT_FILE),
#     path_in_repo=OUTPUT_FILE.name,
#     repo_id=repo_id,
#     repo_type='dataset',
# )
# api.upload_file(
#     path_or_fileobj=str(STATS_FILE),
#     path_in_repo=STATS_FILE.name,
#     repo_id=repo_id,
#     repo_type='dataset',
# )